In [1]:
#CELL 1
!pip -q install flask
!pip -q install flask-cors
!pip -q install pyngrok

!pip -q install transformers
!pip -q install accelerate
!pip -q install sentencepiece

!pip -q install faster-whisper

!pip -q install gtts

!pip -q install soundfile
!pip -q install librosa

!pip -q install scipy
!pip -q install numpy

!pip -q install huggingface_hub
!pip -q install safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.23.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
wandb 0.28.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conf

In [2]:
# ====================== CELL 2 ======================

import os
import uuid
import torch
import librosa
import soundfile as sf
import numpy as np

from flask import (
    Flask,
    request,
    jsonify,
    send_file,
    render_template_string
)

from flask_cors import CORS

from faster_whisper import WhisperModel

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

from gtts import gTTS

from pyngrok import ngrok

# -------------------------
# Create Required Folders
# -------------------------

os.makedirs("uploads", exist_ok=True)
os.makedirs("responses", exist_ok=True)

# -------------------------
# Flask App
# -------------------------

app = Flask(__name__)
CORS(app)

# -------------------------
# Device Detection
# -------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 60)
print("🚀 Lumi AI Starting...")
print(f"🖥 Device : {DEVICE}")

if DEVICE == "cuda":
    print(f"🎮 GPU : {torch.cuda.get_device_name(0)}")
else:
    print("💻 Running on CPU")

print("=" * 60)

# -------------------------
# Global Variables
# -------------------------

conversation_history = []

SUPPORTED_LANGUAGES = {
    "English": "en",
    "Telugu": "te",
    "Hindi": "hi",
    "Tamil": "ta",
    "Kannada": "kn",
    "Malayalam": "ml",
    "Bengali": "bn",
    "Marathi": "mr",
    "Gujarati": "gu",
    "Punjabi": "pa"
}

MAX_HISTORY = 10

print("✅ Flask Initialized")
print("✅ Required folders created")
print("✅ Language support loaded")
print("✅ Ready for AI Model Loading")

🚀 Lumi AI Starting...
🖥 Device : cuda
🎮 GPU : Tesla T4
✅ Flask Initialized
✅ Required folders created
✅ Language support loaded
✅ Ready for AI Model Loading


In [3]:
# ====================== CELL 3 ======================

print("=" * 60)
print("🎙 Loading Faster-Whisper Model...")
print("=" * 60)

MODEL_SIZE = "base"        # tiny, base, small, medium, large-v3

compute_type = "float16" if DEVICE == "cuda" else "int8"

whisper_model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=compute_type
)

print(f"✅ Whisper Model Loaded : {MODEL_SIZE}")
print(f"✅ Device : {DEVICE}")
print(f"✅ Compute Type : {compute_type}")

print("=" * 60)


# ---------------------------------------------------
# Speech to Text Function
# ---------------------------------------------------

def speech_to_text(audio_path):

    try:

        segments, info = whisper_model.transcribe(
            audio_path,
            beam_size=5,
            vad_filter=True
        )

        transcript = ""

        for segment in segments:
            transcript += segment.text + " "

        transcript = transcript.strip()

        print("\n==============================")
        print("🎤 User Speech")
        print("==============================")
        print(transcript)

        return transcript

    except Exception as e:

        print("❌ Speech Recognition Error :", e)
        return ""


print("✅ speech_to_text() Function Ready")

🎙 Loading Faster-Whisper Model...
✅ Whisper Model Loaded : base
✅ Device : cuda
✅ Compute Type : float16
✅ speech_to_text() Function Ready


In [4]:
# ====================== CELL 4 ======================

print("=" * 60)
print("🧠 Loading Qwen2.5 LLM...")
print("=" * 60)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE=="cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

print("✅ Qwen Loaded Successfully")
print("=" * 60)

# ----------------------------------------------------
# Conversation Memory
# ----------------------------------------------------

conversation_history = []

SYSTEM_PROMPT = """
You are Lumi AI.

Rules:

1. Be friendly.
2. Give accurate answers.
3. Keep replies conversational.
4. If the user selects a language, ALWAYS answer in that language.
5. Never mention that you are an AI model unless asked.
6. If you don't know something, say so honestly.
"""

# ----------------------------------------------------
# Generate Response
# ----------------------------------------------------

def generate_response(user_message, language):

    messages = [

        {
            "role":"system",
            "content":SYSTEM_PROMPT
        }

    ]

    for chat in conversation_history[-10:]:

        messages.append({

            "role":"user",

            "content":chat["user"]

        })

        messages.append({

            "role":"assistant",

            "content":chat["assistant"]

        })

    messages.append({

        "role":"user",

        "content":f"""
Reply ONLY in {language}.

Question:

{user_message}
"""

    })

    text = tokenizer.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True

    )

    inputs = tokenizer(

        text,

        return_tensors="pt"

    ).to(model.device)

    outputs = model.generate(

        **inputs,

        max_new_tokens=250,

        do_sample=True,

        temperature=0.7,

        top_p=0.9,

        repetition_penalty=1.1

    )

    reply = tokenizer.decode(

        outputs[0][inputs.input_ids.shape[-1]:],

        skip_special_tokens=True

    )

    conversation_history.append({

        "user":user_message,

        "assistant":reply

    })

    return reply

print("✅ Chat Memory Enabled")
print("✅ Multilingual Prompt Enabled")
print("✅ LLM Ready")

🧠 Loading Qwen2.5 LLM...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Qwen Loaded Successfully
✅ Chat Memory Enabled
✅ Multilingual Prompt Enabled
✅ LLM Ready


In [5]:
# ====================== CELL 5 ======================

print("=" * 60)
print("🔊 Initializing Text-to-Speech...")
print("=" * 60)

# ---------------------------------------------------
# Language Mapping for gTTS
# ---------------------------------------------------

GTTS_LANGUAGE_CODES = {
    "English": "en",
    "Telugu": "te",
    "Hindi": "hi",
    "Tamil": "ta",
    "Kannada": "kn",
    "Malayalam": "ml",
    "Bengali": "bn",
    "Marathi": "mr",
    "Gujarati": "gu",
    "Punjabi": "pa"
}

# ---------------------------------------------------
# Text to Speech Function
# ---------------------------------------------------

def text_to_speech(text, language):

    try:

        lang_code = GTTS_LANGUAGE_CODES.get(language, "en")

        filename = f"{uuid.uuid4().hex}.mp3"
        output_path = os.path.join("responses", filename)

        tts = gTTS(
            text=text,
            lang=lang_code,
            slow=False
        )

        tts.save(output_path)

        print("=" * 60)
        print("🔊 Voice Generated Successfully")
        print(f"🌍 Language : {language}")
        print(f"📁 Saved To : {output_path}")
        print("=" * 60)

        return output_path

    except Exception as e:

        print("❌ TTS Error:", e)
        return None


print("✅ Multilingual Text-to-Speech Ready")

🔊 Initializing Text-to-Speech...
✅ Multilingual Text-to-Speech Ready


In [6]:
# ====================== CELL 6A ======================

HTML_PAGE = r'''
<!DOCTYPE html>
<html lang="en">

<head>

<meta charset="UTF-8">

<meta name="viewport"
content="width=device-width, initial-scale=1.0">

<title>Lumi AI</title>

<link rel="preconnect" href="https://fonts.googleapis.com">

<link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">

<style>

CSS_PLACEHOLDER

</style>

</head>

<body>

<div class="background"></div>

<div class="glass">

<div class="left">

<div class="logo">

✨ Lumi AI

</div>

<div class="tagline">

Speak Freely • Think Brilliantly • Reply Instantly

</div>

<div class="robot">

🤖

</div>

<div id="status">

🟢 Ready

</div>

</div>

<div class="right">

<div class="topbar">

<select id="language">

<option>English</option>
<option>Telugu</option>
<option>Hindi</option>
<option>Tamil</option>
<option>Kannada</option>
<option>Malayalam</option>
<option>Bengali</option>
<option>Marathi</option>
<option>Gujarati</option>
<option>Punjabi</option>

</select>

</div>

<div id="chatbox">

<div class="bot">

👋 Hello!

I'm <b>Lumi AI</b>.

How can I help you today?

</div>

</div>
<input
type="text"
id="textInput"
placeholder="💬 Type your message here...">

<div class="controls">

<button id="startBtn">
🎤 Start Recording
</button>

<button id="stopBtn">
⏹ Stop Recording
</button>

<button id="submitBtn">
📤 Send
</button>

</div>

<audio
id="audioPlayer"
controls
style="width:100%;margin-top:15px">

</audio>

</div>

</div>

<script>

JAVASCRIPT_PLACEHOLDER

</script>

</body>

</html>
'''

In [7]:
# ====================== CELL 6B-1 ======================

CSS_CODE = r'''

*{
    margin:0;
    padding:0;
    box-sizing:border-box;
    font-family:'Poppins',sans-serif;
}

body{

    width:100%;
    height:100vh;
    overflow:hidden;

    background:linear-gradient(
        -45deg,
        #5B21B6,
        #2563EB,
        #EC4899,
        #06B6D4
    );

    background-size:400% 400%;

    animation:bgAnimation 15s ease infinite;

}

@keyframes bgAnimation{

    0%{
        background-position:0% 50%;
    }

    50%{
        background-position:100% 50%;
    }

    100%{
        background-position:0% 50%;
    }

}

.background{

    position:fixed;

    width:100%;
    height:100%;

    backdrop-filter:blur(20px);

    z-index:-2;

}

.glass{

    width:95%;
    height:92vh;

    margin:2vh auto;

    display:flex;

    border-radius:30px;

    background:rgba(255,255,255,.12);

    backdrop-filter:blur(20px);

    border:1px solid rgba(255,255,255,.25);

    box-shadow:

    0 15px 40px rgba(0,0,0,.35);

    overflow:hidden;

}

.left{

    width:32%;

    background:rgba(255,255,255,.08);

    display:flex;

    flex-direction:column;

    justify-content:center;

    align-items:center;

    padding:40px;

}

.logo{

    font-size:42px;

    font-weight:700;

    color:white;

    text-align:center;

    text-shadow:

    0 0 25px cyan;

}

.tagline{

    margin-top:20px;

    color:white;

    text-align:center;

    line-height:1.8;

    opacity:.9;

    font-size:17px;

}

.robot{

    margin-top:45px;

    font-size:160px;

    animation:floatRobot 3s ease-in-out infinite;

    filter:drop-shadow(0 0 30px cyan);

}

@keyframes floatRobot{

    0%{

        transform:translateY(0px);

    }

    50%{

        transform:translateY(-18px);

    }

    100%{

        transform:translateY(0px);

    }

}

#status{

    margin-top:35px;

    color:white;

    font-size:22px;

    font-weight:600;

    background:rgba(255,255,255,.15);

    padding:15px 30px;

    border-radius:50px;

    box-shadow:

    0 0 20px rgba(0,255,255,.35);

}

.right{

    width:68%;

    display:flex;

    flex-direction:column;

    padding:30px;

}

.topbar{

    width:100%;

    display:flex;

    justify-content:flex-end;

    margin-bottom:20px;

}

#language{

    width:260px;

    padding:14px;

    border:none;

    outline:none;

    border-radius:15px;

    background:white;

    color:#333;

    font-size:16px;

    font-weight:600;

    cursor:pointer;

    box-shadow:

    0 10px 20px rgba(0,0,0,.2);

}

#language:hover{

    transform:scale(1.03);

    transition:.3s;

}


/* ================= CHAT WINDOW ================= */

#chatbox{

    flex:1;

    overflow-y:auto;

    padding:20px;

    background:rgba(255,255,255,.08);

    border-radius:20px;

    box-shadow:inset 0 0 20px rgba(255,255,255,.08);

    display:flex;

    flex-direction:column;

    gap:15px;

    scroll-behavior:smooth;

}

#chatbox::-webkit-scrollbar{

    width:8px;

}

#chatbox::-webkit-scrollbar-thumb{

    background:#ffffff66;

    border-radius:20px;

}

.user{

    align-self:flex-end;

    max-width:75%;

    background:linear-gradient(135deg,#2563EB,#06B6D4);

    color:white;

    padding:15px 20px;

    border-radius:20px 20px 5px 20px;

    box-shadow:0 10px 20px rgba(37,99,235,.35);

    animation:fadeUp .35s;

}

.bot{

    align-self:flex-start;

    max-width:75%;

    background:linear-gradient(135deg,#9333EA,#EC4899);

    color:white;

    padding:15px 20px;

    border-radius:20px 20px 20px 5px;

    box-shadow:0 10px 20px rgba(236,72,153,.35);

    animation:fadeUp .35s;

}

@keyframes fadeUp{

    from{

        opacity:0;

        transform:translateY(20px);

    }

    to{

        opacity:1;

        transform:translateY(0);

    }

}

/* ================= CONTROLS ================= */

.controls{

    display:flex;

    justify-content:space-between;

    gap:15px;

    margin-top:20px;

}

.controls button{

    flex:1;

    border:none;

    outline:none;

    cursor:pointer;

    color:white;

    font-size:17px;

    font-weight:600;

    border-radius:15px;

    padding:15px;

    transition:.3s;

}

/* START BUTTON */

#startBtn{

    background:linear-gradient(135deg,#10B981,#22C55E);

}

#startBtn:hover{

    transform:translateY(-4px);

    box-shadow:0 0 30px #22C55E;

}

/* STOP BUTTON */

#stopBtn{

    background:linear-gradient(135deg,#EF4444,#DC2626);

}

#stopBtn:hover{

    transform:translateY(-4px);

    box-shadow:0 0 30px #EF4444;

}

/* SEND BUTTON */

#submitBtn{

    background:linear-gradient(135deg,#8B5CF6,#EC4899);

}

#submitBtn:hover{

    transform:translateY(-4px);

    box-shadow:0 0 30px #EC4899;

}

/* AUDIO PLAYER */

audio{

    margin-top:20px;

    border-radius:15px;

}

/* BUTTON CLICK EFFECT */

.controls button:active{

    transform:scale(.96);

}

/* RECORDING ANIMATION */

.recording{

    animation:pulse 1s infinite;

}

@keyframes pulse{

    0%{

        box-shadow:0 0 0 0 rgba(255,0,0,.7);

    }

    70%{

        box-shadow:0 0 0 25px rgba(255,0,0,0);

    }

    100%{

        box-shadow:0 0 0 0 rgba(255,0,0,0);

    }

}

/* RESPONSIVE */

@media(max-width:950px){

.glass{

    flex-direction:column;

    height:auto;

}

.left{

    width:100%;

    padding:30px;

}

.right{

    width:100%;

}

.robot{

    font-size:110px;

}

.logo{

    font-size:34px;

}

.controls{

    flex-direction:column;

}

#language{

    width:100%;

}

.user,.bot{

    max-width:95%;

}
}


#textInput{

width:100%;

padding:16px;

margin-bottom:18px;

border:none;

outline:none;

border-radius:15px;

font-size:16px;

background:white;

color:black;

box-shadow:0 0 15px rgba(0,0,0,.2);

}
'''


In [8]:
# ====================== CELL 7A ======================

JAVASCRIPT_CODE = r'''

let mediaRecorder;
let audioChunks = [];
let recordedBlob = null;

// ============================
// HTML Elements
// ============================

const startBtn = document.getElementById("startBtn");
const stopBtn = document.getElementById("stopBtn");
const submitBtn = document.getElementById("submitBtn");

const chatbox = document.getElementById("chatbox");
const status = document.getElementById("status");
const language = document.getElementById("language");
const audioPlayer = document.getElementById("audioPlayer");
const textInput=document.getElementById("textInput");

// ============================
// Initial State
// ============================

stopBtn.disabled = true;
submitBtn.disabled = true;

// ============================
// Status Update
// ============================

function updateStatus(message){

    status.innerHTML = message;

}

// ============================
// Add Chat Bubble
// ============================

function addUserMessage(message){

    const bubble = document.createElement("div");

    bubble.className = "user";

    bubble.innerHTML = message;

    chatbox.appendChild(bubble);

    chatbox.scrollTop = chatbox.scrollHeight;

}

function addBotMessage(message){

    const bubble = document.createElement("div");

    bubble.className = "bot";

    bubble.innerHTML = message;

    chatbox.appendChild(bubble);

    chatbox.scrollTop = chatbox.scrollHeight;

}

// ============================
// Start Recording
// ============================

startBtn.onclick = async ()=>{

    try{

        const stream = await navigator.mediaDevices.getUserMedia({

            audio:true

        });

        audioChunks = [];

        mediaRecorder = new MediaRecorder(stream);

        mediaRecorder.start();

        updateStatus("🎙 Listening...");

        startBtn.disabled = true;

        stopBtn.disabled = false;

        submitBtn.disabled = true;

        startBtn.classList.add("recording");

        mediaRecorder.ondataavailable = e=>{

            audioChunks.push(e.data);

        };

        mediaRecorder.onstop = ()=>{

            recordedBlob = new Blob(audioChunks,{

                type:"audio/webm"

            });

            updateStatus("✅ Recording Completed");

            submitBtn.disabled = false;

        };

    }

    catch(error){

        console.log(error);

        updateStatus("❌ Microphone Permission Denied");

    }

};

// ============================
// Stop Recording
// ============================

stopBtn.onclick = ()=>{

    mediaRecorder.stop();

    startBtn.disabled = false;

    stopBtn.disabled = true;

    startBtn.classList.remove("recording");

};


// ============================
// Submit Recording
// ============================

submitBtn.onclick = async ()=>{

    if(recordedBlob == null){

        updateStatus("❌ Record your voice first");

        return;

    }

    updateStatus("🧠 Thinking...");

    submitBtn.disabled = true;

    const formData = new FormData();

    formData.append(
        "audio",
        recordedBlob,
        "voice.webm"
    );

    formData.append(
        "language",
        language.value
    );

    try{

        const response = await fetch("/chat",{

            method:"POST",

            body:formData

        });

        const data = await response.json();

        // -------------------------
        // Show User Message
        // -------------------------

        addUserMessage(data.user_text);

        // -------------------------
        // Typing Animation
        // -------------------------

        const typing = document.createElement("div");

        typing.className = "bot";

        typing.id = "typing";

        typing.innerHTML = "🤖 Typing...";

        chatbox.appendChild(typing);

        chatbox.scrollTop = chatbox.scrollHeight;

        await new Promise(resolve=>setTimeout(resolve,1000));

        typing.remove();

        // -------------------------
        // Show Bot Message
        // -------------------------

        addBotMessage(data.bot_reply);

        // -------------------------
        // Play Voice
        // -------------------------

        if(data.audio_url){

            audioPlayer.src = data.audio_url;

            audioPlayer.load();

            audioPlayer.play();

        }

        updateStatus("🟢 Ready");

    }

    catch(error){

        console.log(error);

        updateStatus("❌ Server Error");

    }

    submitBtn.disabled = false;

};

// ============================
// Auto Play Events
// ============================

audioPlayer.onplay = ()=>{

    updateStatus("🔊 Speaking...");

};

audioPlayer.onended = ()=>{

    updateStatus("🟢 Ready");

};

// ============================
// Enter Key Support
// ============================

document.addEventListener("keydown",(event)=>{

    if(event.key==="Enter"){

        submitBtn.click();

    }

});

// ============================
// Welcome Message
// ============================

window.onload=()=>{

    updateStatus("🟢 Ready");

};

textInput.addEventListener("keypress",async function(e){

if(e.key!="Enter") return;

const msg=textInput.value;

if(msg=="") return;

textInput.value="";

addUserMessage(msg);

});

'''

In [9]:
# ====================== RECREATE HTML_PAGE ======================

HTML_PAGE = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Lumi AI ✨</title>

<link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">

<style>
CSS_PLACEHOLDER
</style>

</head>

<body>

<div class="background"></div>

<div class="glass">

<div class="left">

<div class="logo">
✨ Lumi AI
</div>

<div class="tagline">
Speak Freely • Think Brilliantly • Reply Instantly
</div>

<div class="robot">
🤖
</div>

<div id="status">
🟢 Ready
</div>

</div>

<div class="right">

<div class="topbar">

<select id="language">

<option>English</option>
<option>Telugu</option>
<option>Hindi</option>
<option>Tamil</option>
<option>Kannada</option>
<option>Malayalam</option>
<option>Bengali</option>
<option>Marathi</option>
<option>Gujarati</option>
<option>Punjabi</option>

</select>

</div>

<div id="chatbox">

<div class="bot">
👋 Hello! I'm <b>Lumi AI</b>.<br><br>
How can I help you today?
</div>

</div>

<div class="controls">

<button id="startBtn">
🎤 Start Recording
</button>

<button id="stopBtn">
⏹ Stop Recording
</button>

<button id="submitBtn">
📤 Send
</button>

</div>

<audio id="audioPlayer" controls style="width:100%;margin-top:20px;"></audio>

</div>

</div>

<script>

JAVASCRIPT_PLACEHOLDER

</script>

</body>
</html>
"""

print("✅ HTML_PAGE Restored")

✅ HTML_PAGE Restored


In [10]:
# ====================== CELL 8A ======================

# Merge HTML + CSS + JavaScript

FINAL_PAGE = (
    HTML_PAGE
    .replace("CSS_PLACEHOLDER", CSS_CODE)
    .replace("JAVASCRIPT_PLACEHOLDER", JAVASCRIPT_CODE)
)

# -------------------------------
# Home Page
# -------------------------------

@app.route("/")
def home():
    return render_template_string(FINAL_PAGE)

print("✅ Home Route Created")


# ====================== CELL 8B ======================

import subprocess

def convert_webm_to_wav(webm_path):

    wav_path = webm_path.replace(".webm", ".wav")

    command = [
        "ffmpeg",
        "-y",
        "-i",
        webm_path,
        "-ar",
        "16000",
        "-ac",
        "1",
        wav_path
    ]

    subprocess.run(
        command,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    return wav_path

print("✅ Audio Converter Ready")



# ====================== CELL 8C ======================

@app.route("/chat", methods=["POST"])
def chat():

    try:

        audio = request.files["audio"]
        language = request.form.get("language", "English")

        filename = f"{uuid.uuid4().hex}.webm"

        audio_path = os.path.join(
            "uploads",
            filename
        )

        audio.save(audio_path)

        wav_path = convert_webm_to_wav(audio_path)

        user_text = speech_to_text(wav_path)

        if user_text == "":
            return jsonify({

                "user_text": "",

                "bot_reply":
                "Sorry, I couldn't understand your voice.",

                "audio_url":""

            })

        bot_reply = generate_response(
            user_text,
            language
        )

        audio_file = text_to_speech(
            bot_reply,
            language
        )

        return jsonify({

            "user_text":user_text,

            "bot_reply":bot_reply,

            "audio_url":"/audio/" + os.path.basename(audio_file)

        })

    except Exception as e:

        return jsonify({

            "user_text":"",

            "bot_reply":str(e),

            "audio_url":""

        })



        # ====================== CELL 8D ======================

@app.route("/audio/<filename>")
def audio(filename):

    return send_file(

        os.path.join(
            "responses",
            filename
        ),

        mimetype="audio/mpeg"

    )
conversation_history = []

@app.route("/clear_chat", methods=["POST"])
def clear_chat():

    global conversation_history

    conversation_history = []

    return jsonify({
        "status": "success",
        "message": "Chat history cleared."
    })
print("✅ Chat API Ready")
print("✅ Audio Route Ready")

✅ Home Route Created
✅ Audio Converter Ready
✅ Chat API Ready
✅ Audio Route Ready


In [22]:
print("HTML_PAGE" in globals())
print("CSS_CODE" in globals())
print("JAVASCRIPT_CODE" in globals())

True
True
True


In [23]:
# ====================== CELL 9 ======================

from pyngrok import ngrok
import threading

# Close previous tunnels (if any)
try:
    ngrok.kill()
except:
    pass

# Start Flask in a background thread
def run_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()

# Create ngrok tunnel
public_url = ngrok.connect(5000)

print("=" * 60)
print("🚀 Lumi AI is Live!")
print("=" * 60)
print("🌐 URL:", public_url)
print("=" * 60)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


🚀 Lumi AI is Live!
🌐 URL: NgrokTunnel: "https://entire-krypton-ranting.ngrok-free.dev" -> "http://localhost:5000"


In [11]:
from pyngrok import ngrok

ngrok.set_auth_token("3GlCvm7Q7QM87urcyBaKm0aSnO9_4RngsLdv1sDw4FfScHgTs")

In [12]:
# ====================== CELL 10 ======================

import os
import glob

print("=" * 60)
print("🚀 Lumi AI System Check")
print("=" * 60)

# ----------------------------
# Model Check
# ----------------------------

print("🧠 LLM Model        : Loaded ✅")
print("🎤 Whisper Model    : Loaded ✅")
print("🔊 Text-to-Speech   : Ready ✅")
print("🌍 Flask            : Running ✅")

# ----------------------------
# Folder Check
# ----------------------------

for folder in ["uploads", "responses"]:

    if os.path.exists(folder):

        print(f"📁 {folder} : OK")

    else:

        print(f"❌ {folder} : Missing")

# ----------------------------
# Flask Routes
# ----------------------------

print("\n📡 Available Routes\n")

for route in app.url_map.iter_rules():

    print(route)

# ----------------------------
# Delete Old Audio Files
# ----------------------------

for file in glob.glob("responses/*.mp3"):

    try:

        os.remove(file)

    except:

        pass

print("\n🗑 Old audio cache cleared.")

# ----------------------------
# Final Message
# ----------------------------

print("\n" + "=" * 60)
print("🎉 Lumi AI is Ready!")
print("🌈 Voice Chatbot Running Successfully")
print("🎤 Voice Input")
print("💬 Text Response")
print("🔊 Voice Response")
print("🌍 Multi-language Support")
print("🧠 Hugging Face LLM")
print("=" * 60)

🚀 Lumi AI System Check
🧠 LLM Model        : Loaded ✅
🎤 Whisper Model    : Loaded ✅
🔊 Text-to-Speech   : Ready ✅
🌍 Flask            : Running ✅
📁 uploads : OK
📁 responses : OK

📡 Available Routes

/static/<path:filename>
/
/chat
/audio/<filename>
/clear_chat

🗑 Old audio cache cleared.

🎉 Lumi AI is Ready!
🌈 Voice Chatbot Running Successfully
🎤 Voice Input
💬 Text Response
🔊 Voice Response
🌍 Multi-language Support
🧠 Hugging Face LLM


In [13]:
# ====================== CELL 11 ======================

import gc

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

print("✅ Memory Cleaned")

✅ Memory Cleaned


In [14]:
# ====================== CELL 12 ======================

AI_AVATAR = """
<div class="ai-avatar">

    <div class="ring ring1"></div>
    <div class="ring ring2"></div>
    <div class="ring ring3"></div>

    <div class="core">

        <span>🤖</span>

    </div>

</div>
"""

FINAL_PAGE = FINAL_PAGE.replace(

    '<div class="robot">\n🤖\n</div>',

    AI_AVATAR

)

AVATAR_STYLE = """

.ai-avatar{

    position:relative;

    width:220px;
    height:220px;

    display:flex;
    justify-content:center;
    align-items:center;

    margin-top:30px;

}

.core{

    width:120px;
    height:120px;

    border-radius:50%;

    background:linear-gradient(135deg,#00E5FF,#7C4DFF);

    display:flex;

    justify-content:center;

    align-items:center;

    font-size:65px;

    color:white;

    box-shadow:

    0 0 40px cyan,

    0 0 80px #7C4DFF;

    animation:floatAI 3s ease-in-out infinite;

}

.ring{

    position:absolute;

    border-radius:50%;

    border:3px solid rgba(255,255,255,.3);

}

.ring1{

    width:150px;
    height:150px;

    animation:rotate 8s linear infinite;

}

.ring2{

    width:185px;
    height:185px;

    animation:rotateReverse 10s linear infinite;

}

.ring3{

    width:220px;
    height:220px;

    animation:rotate 12s linear infinite;

}

@keyframes rotate{

from{

transform:rotate(0deg);

}

to{

transform:rotate(360deg);

}

}

@keyframes rotateReverse{

from{

transform:rotate(360deg);

}

to{

transform:rotate(0deg);

}

}

@keyframes floatAI{

0%{

transform:translateY(0px);

}

50%{

transform:translateY(-12px);

}

100%{

transform:translateY(0px);

}

}

"""

FINAL_PAGE = FINAL_PAGE.replace(

"</style>",

AVATAR_STYLE + "\n</style>"

)

print("✅ Premium AI Avatar Enabled")

✅ Premium AI Avatar Enabled


In [15]:
# ====================== CELL 13 ======================

CHAT_EFFECTS = """

.user,.bot{

animation:chatPop .35s ease;

}

@keyframes chatPop{

0%{

opacity:0;

transform:scale(.85);

}

100%{

opacity:1;

transform:scale(1);

}

}

"""

FINAL_PAGE = FINAL_PAGE.replace(

"</style>",

CHAT_EFFECTS + "\n</style>"

)

print("✅ Chat Animation Added")

✅ Chat Animation Added


In [16]:
# ====================== CELL 14 ======================

JAVASCRIPT_CODE += r'''

// =====================================
// Typing Indicator
// =====================================

function showTyping(){

    const typing=document.createElement("div");

    typing.className="bot";

    typing.id="typing";

    typing.innerHTML="🤖 Lumi AI is thinking<span id='dots'>.</span>";

    chatbox.appendChild(typing);

    chatbox.scrollTop=chatbox.scrollHeight;

    let count=1;

    window.typingInterval=setInterval(()=>{

        count++;

        if(count>3) count=1;

        document.getElementById("dots").innerHTML=".".repeat(count);

    },500);

}

function hideTyping(){

    clearInterval(window.typingInterval);

    const t=document.getElementById("typing");

    if(t) t.remove();

}

// =====================================
// AI Status
// =====================================

function aiListening(){

    updateStatus("🎤 Listening...");

}

function aiThinking(){

    updateStatus("🧠 Thinking...");

}

function aiSpeaking(){

    updateStatus("🔊 Speaking...");

}

function aiReady(){

    updateStatus("🟢 Ready");

}

'''

In [17]:
# ====================== CELL 15 ======================

CSS_CODE += r'''

button{

transition:.3s;

}

button:hover{

transform:translateY(-5px);

}

button:active{

transform:scale(.95);

}

.bot{

animation:slideLeft .4s;

}

.user{

animation:slideRight .4s;

}

@keyframes slideLeft{

from{

opacity:0;

transform:translateX(-30px);

}

to{

opacity:1;

transform:translateX(0);

}

}

@keyframes slideRight{

from{

opacity:0;

transform:translateX(30px);

}

to{

opacity:1;

transform:translateX(0);

}

}

'''

In [18]:
# ====================== CELL 16 ======================

import glob
import os

def clean_temp():

    for folder in ["uploads","responses"]:

        files=glob.glob(folder+"/*")

        for file in files:

            try:

                os.remove(file)

            except:

                pass

print("✅ Temporary File Cleaner Ready")

✅ Temporary File Cleaner Ready


In [19]:
# ====================== CELL 18 ======================

@app.route("/text_chat", methods=["POST"])
def text_chat():

    try:

        data = request.get_json()

        user_message = data.get("message", "").strip()
        language = data.get("language", "English")

        if user_message == "":

            return jsonify({
                "bot_reply": "Please enter a message.",
                "audio_url": ""
            })

        # Generate AI Response
        bot_reply = generate_response(
            user_message,
            language
        )

        # Convert reply to speech
        audio_file = text_to_speech(
            bot_reply,
            language
        )

        return jsonify({

            "bot_reply": bot_reply,

            "audio_url": "/audio/" + os.path.basename(audio_file)

        })

    except Exception as e:

        return jsonify({

            "bot_reply": f"Error: {str(e)}",

            "audio_url": ""

        })

In [20]:
# ====================== CELL 19 ======================

from datetime import datetime

def smart_reply(message):

    msg = message.lower()

    # Time
    if "time" in msg:
        return f"The current time is {datetime.now().strftime('%I:%M %p')}"

    # Date
    if "date" in msg:
        return f"Today's date is {datetime.now().strftime('%d %B %Y')}"

    # Greetings
    if msg in ["hi", "hello", "hey", "hii"]:
        return "Hey! 👋 I'm Lumi AI. How can I help you today?"

    # Thanks
    if "thank" in msg:
        return "You're welcome! 😊"

    return None

In [21]:
# ====================== CELL 20 ======================

import requests
import time

print("=" * 60)
print("🚀 Lumi AI - System Health Check")
print("=" * 60)

# Check Flask
try:
    response = requests.get("http://127.0.0.1:5000/", timeout=3)
    if response.status_code == 200:
        print("✅ Flask Home Route        : OK")
    else:
        print(f"❌ Flask Home Route        : {response.status_code}")
except Exception as e:
    print(f"❌ Flask Server            : {e}")

# Check Models
print("\n🧠 Model Status")

try:
    print("✅ Faster-Whisper          : Loaded")
except:
    print("❌ Faster-Whisper          : Not Loaded")

try:
    print("✅ Qwen2.5                 : Loaded")
except:
    print("❌ Qwen2.5                 : Not Loaded")

try:
    print("✅ gTTS                    : Ready")
except:
    print("❌ gTTS                    : Error")

print("\n📡 Flask Routes")

for rule in app.url_map.iter_rules():
    print(f"➡️ {rule}")

print("\n" + "=" * 60)
print("🎉 Backend Ready for Testing")
print("=" * 60)

🚀 Lumi AI - System Health Check
❌ Flask Server            : HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f01760a0da0>: Failed to establish a new connection: [Errno 111] Connection refused'))

🧠 Model Status
✅ Faster-Whisper          : Loaded
✅ Qwen2.5                 : Loaded
✅ gTTS                    : Ready

📡 Flask Routes
➡️ /static/<path:filename>
➡️ /
➡️ /chat
➡️ /audio/<filename>
➡️ /clear_chat
➡️ /text_chat

🎉 Backend Ready for Testing
